# Gemma 4 — Basic Usage

## Imports

In [1]:
from pprint import pprint

import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("MPS (Apple Silicon GPU) available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

torch: 2.13.0
transformers: 5.14.1
MPS (Apple Silicon GPU) available: True
CUDA available: False


## Load Model and Processor

In [2]:
MODEL_ID = "google/gemma-4-E2B-it"

processor = transformers.AutoProcessor.from_pretrained(MODEL_ID)
model = transformers.AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)

print(f"Architecture: {model.config.architectures}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {model.device}")
print(f"Dtype: {model.dtype}")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Architecture: ['Gemma4ForConditionalGeneration']
Parameters: 5,104,297,504
Device: mps:0
Dtype: torch.bfloat16


## Single Turn Generation

In [3]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

chat = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(chat)

<bos><|turn>user
What is the capital of France?<turn|>
<|turn>model



In [4]:
inputs = processor(text=chat, return_tensors="pt", add_special_tokens=False).to(model.device)
pprint(inputs, sort_dicts=False, width=120)

{'input_ids': tensor([[     2,    105,   2364,    107,   3689,    563,    506,   5279,    529,
           7001, 236881,    106,    107,    105,   4368,    107]],
       device='mps:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0'),
 'mm_token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], device='mps:0')}


In [5]:
outputs = model.generate(**inputs, max_new_tokens=128)
print(outputs)

tensor([[     2,    105,   2364,    107,   3689,    563,    506,   5279,    529,
           7001, 236881,    106,    107,    105,   4368,    107,    818,   5279,
            529,   7001,    563,   5213,  50429,  84750,    106]],
       device='mps:0')


In [6]:
print(processor.decode(outputs[0]))

<bos><|turn>user
What is the capital of France?<turn|>
<|turn>model
The capital of France is **Paris**.<turn|>


In [7]:
input_len = inputs["input_ids"].shape[-1]
response = processor.decode(outputs[0][input_len:])
print(response)

The capital of France is **Paris**.<turn|>


In [8]:
response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
print(response)

The capital of France is **Paris**.


## System Prompt

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who responds in all capitals."},
    {"role": "user", "content": "What is the capital of France?"},
]

chat = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print(chat)

<bos><|turn>system
You are a helpful assistant who responds in all capitals.<turn|>
<|turn>user
What is the capital of France?<turn|>
<|turn>model



In [10]:
inputs = processor(text=chat, return_tensors="pt", add_special_tokens=False).to(model.device)
input_len = inputs["input_ids"].shape[-1]
outputs = model.generate(**inputs, max_new_tokens=128)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

PARIS


## Thinking Generation

In [14]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

chat = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

print(chat)

<bos><|turn>system
<|think|>
<turn|>
<|turn>user
What is the capital of France?<turn|>
<|turn>model



In [16]:
inputs = processor(text=chat, return_tensors="pt", add_special_tokens=False).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=1024)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

thought
Thinking Process:

1.  **Analyze the Request:** The user is asking for "the capital of France."
2.  **Identify the Knowledge Domain:** This is a factual geography question.
3.  **Retrieve the Information:** I need to recall the capital city of France.
    *   *Recall:* Paris is implicate the capital.
4.  **Formulate the Answer:** State the answer clearly and directly.
5.  **Final Check:** Does the answer directly address the question? Yes. (The capital of France is Paris.)The capital of France is **Paris**.


## Multi-Turn Generation

In [18]:
messages = [
    {"role": "user", "content": "What's the capital of France?"},
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False,
    return_dict=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

The capital of Francelinux is **Paris**.


In [19]:
messages.append({"role": "assistant", "content": response})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant', 'content': 'The capital of Francelinux is **Paris**.'}]


In [20]:
prompt = "What is a famous landmark there?"

messages.append({"role": "user", "content": prompt})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant', 'content': 'The capital of Francelinux is **Paris**.'},
 {'role': 'user', 'content': 'What is a famous landmark there?'}]


In [21]:
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False,
    return_dict=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

One of the most famous landmarks in Paris is the **Eiffel Tower**.


## Streaming Generation

In [22]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
    enable_thinking=False,
    return_dict=True,
).to(model.device)

streamer = transformers.TextStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)
outputs = model.generate(**inputs, max_new_tokens=128, streamer=streamer)

The capital of France is **Paris**.
